# Lab 2: Audit data and evaluate one policy

**Goal:** Check one complete demonstration, the fixed episode split, the actual training statistics, and a live ACT result. A classical controller also needs a known sensor layout, action convention, and test condition before its error is useful.

**Inputs:** The pinned LIBERO training files, the saved bowl-plate ACT bundle, and the fixed simulator states. **Compute:** CPU for the data audit; local GPU and EGL for the last live cell. **Cost:** US$0 on this prepared local machine. The selected training files use several GiB of disk. The reference path takes several minutes, including two live rollouts.

Run from the repository root after `.venv/bin/robot-lab data fetch --scope training`. Equivalent CLI for the live cell: `.venv/bin/robot-lab eval --checkpoint .local-data/runs/act-bowl-plate-probe-200/checkpoint-000200 --state-id 0 --state-id 1 --output .local-data/runs/lab2-cli-eval`.


## 1. Read the fixed split

The dataset has complete episodes. We divide by episode, so adjacent frames of one demonstration cannot enter both training and validation. The statistics file names the same split hash and dataset revision. Simulator development states 0–19 are a separate evaluation set; final states 20–49 stay closed during model selection.


In [ ]:
from pathlib import Path
import os
import json
import tempfile
import yaml
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Video

WORKSPACE = Path.cwd().resolve()
assert (WORKSPACE / "configs" / "data" / "episodes.yaml").is_file(), "Start Jupyter from the repository root."
DATA_ROOT = WORKSPACE / ".local-data"
os.environ["HF_HUB_CACHE"] = str(DATA_ROOT / "hf" / "hub")
os.environ["HF_DATASETS_CACHE"] = str(DATA_ROOT / "hf" / "datasets")
TASK = "bowl_plate"
from vla_lab.data.inspection import audit_task_split, audit_example
from vla_lab.evaluation.states import selected_state_ids

audit = audit_task_split(WORKSPACE / "configs", TASK)
split = yaml.safe_load((WORKSPACE / "configs" / "data" / "episodes.yaml").read_text())["tasks"][TASK]
train_ids = split["train_episode_ids"]
validation_ids = split["validation_episode_ids"]
selected_state_ids(WORKSPACE / "configs", TASK, "development", [0, 1])
print(json.dumps(audit, indent=2))
print("First train IDs:", train_ids[:5], "First validation IDs:", validation_ids[:5])

## 2. Decode a full demonstration

LeRobot v3 can keep many episodes in one Parquet or video file. Its episode metadata gives the boundaries. The sample loader decodes actual AV1 camera frames. The graph uses stored timestamps as **labels**. They do not prove the simulator's control period. The live controller uses a measured 20 Hz tick; this dataset declares 10 Hz. We do not resample its actions to hide that difference.


In [ ]:
from vla_lab.training.act import _frozen_training_dataset

dataset = _frozen_training_dataset(TASK, WORKSPACE / "configs", DATA_ROOT)
sample = dataset[0]
example = audit_example(sample)
assert example["episode_index"] == train_ids[0]
print(json.dumps(example, indent=2))

rows = dataset.hf_dataset
first_episode = train_ids[0]
indices = [i for i, episode in enumerate(rows["episode_index"]) if episode == first_episode]
frames = np.asarray([rows[i]["frame_index"] for i in indices])
times = np.asarray([rows[i]["timestamp"] for i in indices])
actions = np.asarray([rows[i]["action"] for i in indices])
assert np.array_equal(frames, np.arange(len(frames)))
assert np.all(np.diff(times) > 0)
assert len(indices) > 20
print("Complete episode:", first_episode, "frames:", len(indices), "stored fps:", dataset.fps)
print("Median stored timestamp step (s):", float(np.median(np.diff(times))))
print("Action range:", float(actions.min()), float(actions.max()))

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
axes[0].imshow(sample["observation.images.image"].permute(1, 2, 0).numpy())
axes[0].set_title("Workspace camera")
axes[1].imshow(sample["observation.images.image2"].permute(1, 2, 0).numpy())
axes[1].set_title("Wrist camera")
for axis in axes[:2]: axis.axis("off")
for column, label in enumerate(("x", "y", "z")):
    axes[2].plot(frames, actions[:, column], label=label)
axes[2].set(xlabel="Stored frame index", ylabel="Relative action value", title="One complete episode")
axes[2].legend()
plt.tight_layout()
plt.show()

## 3. Check the training processor

Only training episodes contribute state and action statistics. Check one state value before and after the saved processor. The image encoder uses its selected fixed image normalization path. A validation episode must not change these statistics.


In [ ]:
import torch
from torch.utils.data._utils.collate import default_collate
from vla_lab.policy.act import make_act_config, make_act_processors

stats = json.loads((WORKSPACE / "configs" / "data" / "stats" / f"{TASK}.json").read_text())
config = make_act_config(device="cpu")
preprocessor, _ = make_act_processors(config, TASK, WORKSPACE / "configs")
processed = preprocessor(default_collate([sample]))
raw_state = sample["observation.state"].numpy()
mean = np.asarray(stats["modalities"]["observation.state"]["mean"])
std = np.asarray(stats["modalities"]["observation.state"]["std"])
expected = (raw_state - mean) / std
actual = processed["observation.state"][0].detach().cpu().numpy()
assert np.allclose(actual, expected, atol=1e-5)
print("Raw state:", raw_state.round(3).tolist())
print("Processed state:", actual.round(3).tolist())
print("Stats count:", stats["modalities"]["observation.state"]["count"])
print("Split hash:", stats["split_sha256"])

## 4. Compare storage formats with a small sample

The HDF5 file below is a local teaching example from three rows. It is **not** a new training dataset. RLDS/Open X-Embodiment represents data as episodes with ordered steps; the small Python record below illustrates a three-step prefix. Its final sampled step is not the end of the complete episode. An actual conversion also needs a documented state schema, action units, camera names, task text, and time base. Read the [Open X-Embodiment data format notes](https://github.com/google-deepmind/open_x_embodiment/blob/main/README.md) and the [LeRobot v3 format](https://huggingface.co/docs/lerobot/lerobot-dataset-v3) for 20 minutes. Predict what goes wrong if a conversion marks a relative action as an absolute target.


In [ ]:
import h5py

with tempfile.TemporaryDirectory() as temp_dir:
    path = Path(temp_dir) / "three-steps.h5"
    with h5py.File(path, "w") as file:
        group = file.create_group(f"episode_{first_episode}")
        group.create_dataset("state", data=np.asarray([rows[i]["observation.state"] for i in indices[:3]]))
        group.create_dataset("action", data=actions[:3])
        group.create_dataset("timestamp", data=times[:3])
        group.attrs["action_convention"] = "LIBERO relative controller values"
    with h5py.File(path) as file:
        assert file[f"episode_{first_episode}/action"].shape == (3, 7)
        print("HDF5 groups:", list(file), "action shape:", file[f"episode_{first_episode}/action"].shape)

rlds_example = {"episode_id": first_episode, "steps": [
    {"observation": {"state": rows[i]["observation.state"]},
     "action": rows[i]["action"], "timestamp": rows[i]["timestamp"],
     "is_first": offset == 0, "is_last": False}
    for offset, i in enumerate(indices[:3])
]}
assert rlds_example["steps"][0]["is_first"] and not any(step["is_last"] for step in rlds_example["steps"])
print("Small RLDS-like example:", len(rlds_example["steps"]), "ordered steps")

## 5. Run a fixed live development evaluation

The saved P0 reference succeeded on 19 of 20 bowl-plate development states. That result is an existing reference run, not a new measurement here. Run two exact states now. The evaluator saves each outcome and video, including failed attempts. One or two states are a wiring check, not a performance estimate.


In [ ]:
from vla_lab.evaluation.states import evaluate_act_states

reference = json.loads((WORKSPACE / "artifacts" / "reference" / "act" / TASK / "summary.json").read_text())
checkpoint = WORKSPACE / ".local-data" / "runs" / "act-bowl-plate-probe-200" / "checkpoint-000200"
runs_root = DATA_ROOT / "runs"
runs_root.mkdir(parents=True, exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix="lab2-eval-", dir=runs_root))
result = evaluate_act_states(
    checkpoint=checkpoint,
    configs_root=WORKSPACE / "configs",
    data_root=DATA_ROOT,
    output_dir=run_dir,
    state_ids=[0, 1],
)
print("Existing 20-state development reference:", reference["successes"], "/", reference["episodes"])
print("New two-state live run:", json.dumps(result, indent=2))
records = [json.loads(line) for line in (run_dir / "episodes.jsonl").read_text().splitlines()]
assert [row["initial_state_id"] for row in records] == [0, 1]
print("Per-state reasons:", [(row["initial_state_id"], row["termination_reason"]) for row in records])
video = records[0]["artifact_paths"].get("video")
if video: display(Video(video, embed=True, html_attributes="controls"))

## Exercise and completion check

**Exercise:** Select a different complete *training* episode and audit its frame sequence and stored timestamp steps. Change only that episode ID. Keep the split and the evaluation state IDs fixed. Write down one conversion error that would invalidate action labels. See [exercise instructions](exercise.md) and [the separate solution](../../solutions/02-data-evaluation.md).

You are done when you have two camera views, a complete frame sequence, a matching train-only processor value, a small HDF5 and RLDS-like record, and two saved live evaluation rows. A missing AV1 decoder stops the data cell. A missing GPU or EGL runtime stops the live cell. This lab does not prove that the two selected simulator states represent all task starts, or that dataset timestamps equal controller time.
